In [19]:
import pandas as pd
from IPython.display import display

TOP_N_DIAGNOSES = 5
AFIB_ICD10_PREFIX = "I48"
AFIB_ICD9_PREFIX = "4273"
CHUNK_SIZE = 500_000

paths = {
    "admissions": "/data/padmalab_external/special_project/physionet.org/files/mimiciv/3.1/hosp/admissions.csv.gz",
    "edstays": "/data/padmalab_external/special_project/physionet.org/files/mimic-iv-ed/2.2/ed/edstays.csv.gz",
    "hosp_diagnoses_icd": "/data/padmalab_external/special_project/physionet.org/files/mimiciv/3.1/hosp/diagnoses_icd.csv.gz",
    "ed_diagnoses_icd": "/data/padmalab_external/special_project/physionet.org/files/mimic-iv-ed/2.2/ed/diagnosis.csv.gz",
    "ecg": "/data/padmalab_external/special_project/physionet.org/files/mimic-iv-ecg/1.0/record_list.csv",
    "censor": "/data/padmalab_external/special_project/Weijie_Code/Discharged_ECG_Death_label/mimic_censor_death_label.csv.gz",
}


def afib_mask(df: pd.DataFrame) -> pd.Series:
    icd_version = pd.to_numeric(df["icd_version"], errors="coerce")
    seq_num = pd.to_numeric(df["seq_num"], errors="coerce")
    icd_code = (
        df["icd_code"]
        .fillna("")
        .astype("string")
        .str.upper()
        .str.replace(".", "", regex=False)
        .str.strip()
    )
    return seq_num.le(TOP_N_DIAGNOSES) & (
        ((icd_version == 10) & icd_code.str.startswith(AFIB_ICD10_PREFIX))
        | ((icd_version == 9) & icd_code.str.startswith(AFIB_ICD9_PREFIX))
    )


def collect_afib_hits(path: str, join_cols: list[str]) -> pd.DataFrame:
    hits = []
    for chunk in pd.read_csv(
        path,
        usecols=join_cols + ["seq_num", "icd_code", "icd_version"],
        chunksize=CHUNK_SIZE,
        dtype={"icd_code": "string"},
    ):
        mask = afib_mask(chunk)
        if mask.any():
            hits.append(chunk.loc[mask, join_cols].drop_duplicates())

    if not hits:
        return pd.DataFrame(columns=join_cols)

    return pd.concat(hits, ignore_index=True).drop_duplicates()


admissions = pd.read_csv(
    paths["admissions"],
    usecols=["subject_id", "hadm_id", "admittime"],
    parse_dates=["admittime"],
)
edstays = pd.read_csv(
    paths["edstays"],
    usecols=["subject_id", "stay_id", "intime"],
    parse_dates=["intime"],
)

hosp_hits = collect_afib_hits(paths["hosp_diagnoses_icd"], ["subject_id", "hadm_id"])
ed_hits = collect_afib_hits(paths["ed_diagnoses_icd"], ["subject_id", "stay_id"])

hosp_events = (
    hosp_hits.merge(admissions, on=["subject_id", "hadm_id"], how="left")
    .rename(columns={"admittime": "afib_onset_time"})
)
hosp_events["source"] = "hosp"
hosp_events["encounter_id"] = hosp_events["hadm_id"].astype("Int64").astype("string")

ed_events = (
    ed_hits.merge(edstays, on=["subject_id", "stay_id"], how="left")
    .rename(columns={"intime": "afib_onset_time"})
)
ed_events["source"] = "ed"
ed_events["encounter_id"] = ed_events["stay_id"].astype("Int64").astype("string")

afib_events = pd.concat(
    [
        hosp_events[["subject_id", "afib_onset_time", "source", "encounter_id"]],
        ed_events[["subject_id", "afib_onset_time", "source", "encounter_id"]],
    ],
    ignore_index=True,
).dropna(subset=["afib_onset_time"])
afib_events["afib_onset_date"] = afib_events["afib_onset_time"].dt.normalize()

afib_patients = (
    afib_events.sort_values(["subject_id", "afib_onset_time", "source"])
    .drop_duplicates(subset="subject_id", keep="first")
    .reset_index(drop=True)
)

ecg_df = pd.read_csv(
    paths["ecg"],
    usecols=["subject_id", "study_id", "ecg_time", "path"],
    parse_dates=["ecg_time"],
)

ecgs_with_afib = ecg_df.merge(
    afib_patients[["subject_id", "afib_onset_time", "afib_onset_date"]],
    on="subject_id",
    how="inner",
).dropna(subset=["ecg_time"])

afib_ecgs_on_or_after_onset = ecgs_with_afib.loc[
    ecgs_with_afib["ecg_time"] >= ecgs_with_afib["afib_onset_time"]
].copy()

afib_ecgs = ecgs_with_afib.loc[
    ecgs_with_afib["ecg_time"].dt.normalize() > ecgs_with_afib["afib_onset_date"]
].copy()

print(f"HOSP AFib encounters: {len(hosp_hits):,}")
print(f"ED AFib encounters: {len(ed_hits):,}")
print(f"AFib patients: {len(afib_patients):,}")
print(f"ECGs on/after onset time: {len(afib_ecgs_on_or_after_onset):,}")
print(f"ECGs after first AFib day: {len(afib_ecgs):,}")

display(afib_patients.head())
display(afib_ecgs[["subject_id", "study_id", "ecg_time", "afib_onset_date", "path"]].head())


HOSP AFib encounters: 36,296
ED AFib encounters: 7,524
AFib patients: 21,213
ECGs on/after onset time: 132,998
ECGs after first AFib day: 123,516


,subject_id,afib_onset_time,source,encounter_id,afib_onset_date
0,10001667,2173-08-22 17:16:00,hosp,22672901,2173-08-22
1,10001877,2149-05-21 15:53:00,hosp,25679292,2149-05-21
2,10001884,2130-04-08 22:06:00,hosp,29675586,2130-04-08
3,10002131,2128-03-17 14:53:00,hosp,24065018,2128-03-17
4,10002430,2125-06-23 09:00:00,hosp,27218502,2125-06-23


,subject_id,study_id,ecg_time,afib_onset_date,path
1,10001877,41316269,2149-05-25 02:44:00,2149-05-21,files/p1000/p10001877/s41316269/41316269
2,10001877,46566322,2150-11-21 18:48:00,2149-05-21,files/p1000/p10001877/s46566322/46566322
3,10001877,42315368,2150-11-21 23:10:00,2149-05-21,files/p1000/p10001877/s42315368/42315368
34,10001884,49736004,2130-04-15 16:55:00,2130-04-08,files/p1000/p10001884/s49736004/49736004
35,10001884,40473394,2130-04-15 20:16:00,2130-04-08,files/p1000/p10001884/s40473394/40473394


In [20]:
BLEEDING_CODE_MAP = {
    "Intracranial hemorrhage": {
        "ICD10": "I60, I61, I62",
        "ICD9": "430, 431, 432.0, 432.1, 432.9",
    },
    "Gastrointestinal bleeding": {
        "ICD10": "I8501, I8511, K2211, K250, K252, K254, K256, K260, K262, K264, K266, K270, K272, K274, K276, K280, K282, K284, K286, K2901, K2921, K2931, K2941, K2951, K2961, K2971, K2981, K2991, K31811, K3182, K5521, K625, K920, K921, K922",
        "ICD9": "456.0, 456.20, 531.0, 531.2, 531.4, 531.6, 532.0, 532.2, 532.4, 532.6, 533.0, 533.2, 533.4, 533.6, 534.0, 534.2, 534.4, 534.6, 535.01, 535.11, 535.31, 535.41, 535.51, 535.61, 535.71, 535.81, 535.91, 569.3, 578.0, 578.1, 578.9",
    },
    "Other major bleeding (non-GI / non-intracranial)": {
        "ICD10": "K661, I312, J942, M250, M2500, M2508, N02, R04, R041, R042, R048, R049, D62, H210, H313, H356, H431, R58",
        "ICD9": "568.81, 423.0, 860.2, 719.1x, 786.3, 784.7, 379.23",
    },
}

OUTCOME_SLUGS = {
    "Intracranial hemorrhage": "intracranial_hemorrhage",
    "Gastrointestinal bleeding": "gastrointestinal_bleeding",
    "Other major bleeding (non-GI / non-intracranial)": "other_major_bleeding",
}


In [21]:
import numpy as np


def normalize_prefix(code: str) -> str:
    code = str(code).upper().replace(".", "").strip()
    if code.endswith("X"):
        code = code[:-1]
    return code


NORMALIZED_BLEEDING_CODES = {
    outcome_name: {
        10: tuple(normalize_prefix(code) for code in code_spec["ICD10"].split(",") if code.strip()),
        9: tuple(normalize_prefix(code) for code in code_spec["ICD9"].split(",") if code.strip()),
    }
    for outcome_name, code_spec in BLEEDING_CODE_MAP.items()
}


def collect_outcome_hits(path: str, join_cols: list[str], subject_filter: set[int]) -> pd.DataFrame:
    hits = []
    for chunk in pd.read_csv(
        path,
        usecols=join_cols + ["icd_code", "icd_version"],
        chunksize=CHUNK_SIZE,
        dtype={"icd_code": "string"},
    ):
        chunk["subject_id"] = pd.to_numeric(chunk["subject_id"], errors="coerce")
        chunk = chunk[chunk["subject_id"].isin(subject_filter)].copy()
        if chunk.empty:
            continue

        chunk["subject_id"] = chunk["subject_id"].astype("int64")
        icd_version = pd.to_numeric(chunk["icd_version"], errors="coerce")
        icd_code = (
            chunk["icd_code"]
            .fillna("")
            .astype("string")
            .str.upper()
            .str.replace(".", "", regex=False)
            .str.strip()
        )

        for outcome_name, code_map in NORMALIZED_BLEEDING_CODES.items():
            mask = (
                ((icd_version == 10) & icd_code.str.startswith(code_map[10]))
                | ((icd_version == 9) & icd_code.str.startswith(code_map[9]))
            )
            if mask.any():
                matched = chunk.loc[mask, join_cols].copy()
                matched["outcome_name"] = outcome_name
                hits.append(matched.drop_duplicates())

    if not hits:
        return pd.DataFrame(columns=join_cols + ["outcome_name"])

    out = pd.concat(hits, ignore_index=True).drop_duplicates()
    out["subject_id"] = out["subject_id"].astype("int64")
    return out


def next_event_times(index_df: pd.DataFrame, event_df: pd.DataFrame) -> pd.Series:
    next_times = pd.Series(pd.NaT, index=index_df.index, dtype="datetime64[ns]")
    event_lookup = {
        subject_id: grp["event_time"].sort_values().to_numpy(dtype="datetime64[ns]")
        for subject_id, grp in event_df.groupby("subject_id", sort=False)
    }

    for subject_id, row_idx in index_df.groupby("subject_id", sort=False).groups.items():
        candidate_events = event_lookup.get(subject_id)
        if candidate_events is None or len(candidate_events) == 0:
            continue

        row_idx = np.asarray(row_idx)
        ecg_times = index_df.loc[row_idx, "ecg_time"].to_numpy(dtype="datetime64[ns]")
        censor_times = index_df.loc[row_idx, "censor_time"].to_numpy(dtype="datetime64[ns]")
        positions = np.searchsorted(candidate_events, ecg_times, side="right")
        has_future_event = positions < len(candidate_events)
        if not has_future_event.any():
            continue

        matched_rows = row_idx[has_future_event]
        matched_events = candidate_events[positions[has_future_event]]
        within_censor = matched_events <= censor_times[has_future_event]
        if within_censor.any():
            next_times.loc[matched_rows[within_censor]] = pd.to_datetime(matched_events[within_censor])

    return next_times


subject_filter = set(
    pd.to_numeric(afib_ecgs["subject_id"], errors="coerce").dropna().astype("int64").unique()
)

admissions_for_events = admissions.copy()
admissions_for_events["subject_id"] = pd.to_numeric(
    admissions_for_events["subject_id"], errors="coerce"
).astype("int64")

edstays_for_events = edstays.copy()
edstays_for_events["subject_id"] = pd.to_numeric(
    edstays_for_events["subject_id"], errors="coerce"
).astype("int64")

hosp_bleeding_hits = collect_outcome_hits(
    paths["hosp_diagnoses_icd"],
    ["subject_id", "hadm_id"],
    subject_filter,
)
ed_bleeding_hits = collect_outcome_hits(
    paths["ed_diagnoses_icd"],
    ["subject_id", "stay_id"],
    subject_filter,
)

hosp_bleeding_events = hosp_bleeding_hits.merge(
    admissions_for_events[["subject_id", "hadm_id", "admittime"]],
    on=["subject_id", "hadm_id"],
    how="left",
).rename(columns={"admittime": "event_time"})
hosp_bleeding_events["event_source"] = "hosp"
hosp_bleeding_events["encounter_id"] = hosp_bleeding_events["hadm_id"].astype("Int64").astype("string")

ed_bleeding_events = ed_bleeding_hits.merge(
    edstays_for_events[["subject_id", "stay_id", "intime"]],
    on=["subject_id", "stay_id"],
    how="left",
).rename(columns={"intime": "event_time"})
ed_bleeding_events["event_source"] = "ed"
ed_bleeding_events["encounter_id"] = ed_bleeding_events["stay_id"].astype("Int64").astype("string")

outcome_events = pd.concat(
    [
        hosp_bleeding_events[["subject_id", "outcome_name", "event_time", "event_source", "encounter_id"]],
        ed_bleeding_events[["subject_id", "outcome_name", "event_time", "event_source", "encounter_id"]],
    ],
    ignore_index=True,
).dropna(subset=["event_time"]).drop_duplicates()
outcome_events["subject_id"] = outcome_events["subject_id"].astype("int64")

censor_df = pd.read_csv(paths["censor"], compression="gzip").rename(
    columns={"patient_id": "subject_id", "death_event": "censor_death_event"}
)
censor_df["subject_id"] = pd.to_numeric(censor_df["subject_id"], errors="coerce")
censor_df = censor_df.dropna(subset=["subject_id"]).copy()
censor_df["subject_id"] = censor_df["subject_id"].astype("int64")
censor_df["censor_time"] = pd.to_datetime(censor_df["censor_time"], errors="coerce")
censor_df["censor_death_event"] = (
    censor_df["censor_death_event"].astype(str).str.strip().str.lower() == "true"
)

index_ecgs = afib_ecgs.copy()
index_ecgs["subject_id"] = pd.to_numeric(index_ecgs["subject_id"], errors="coerce")
index_ecgs = index_ecgs.dropna(subset=["subject_id"]).copy()
index_ecgs["subject_id"] = index_ecgs["subject_id"].astype("int64")
index_ecgs = index_ecgs.merge(
    censor_df[["subject_id", "censor_time", "censor_death_event"]],
    on="subject_id",
    how="left",
)

missing_censor_count = int(index_ecgs["censor_time"].isna().sum())
no_followup_count = int((index_ecgs["ecg_time"] >= index_ecgs["censor_time"]).sum())

afib_ecg_labels = index_ecgs.loc[
    index_ecgs["censor_time"].notna() & (index_ecgs["ecg_time"] < index_ecgs["censor_time"])
].copy().reset_index(drop=True)

for outcome_name, slug in OUTCOME_SLUGS.items():
    outcome_subset = outcome_events.loc[
        outcome_events["outcome_name"] == outcome_name,
        ["subject_id", "event_time"],
    ].copy()
    afib_ecg_labels[f"{slug}_date"] = next_event_times(afib_ecg_labels, outcome_subset)
    afib_ecg_labels[f"{slug}_label"] = afib_ecg_labels[f"{slug}_date"].notna()
    afib_ecg_labels[f"{slug}_time"] = (
        afib_ecg_labels[f"{slug}_date"].fillna(afib_ecg_labels["censor_time"]) - afib_ecg_labels["ecg_time"]
    ).dt.days

afib_ecg_labels["death_date"] = afib_ecg_labels["censor_time"].where(
    afib_ecg_labels["censor_death_event"] & afib_ecg_labels["censor_time"].gt(afib_ecg_labels["ecg_time"])
)
afib_ecg_labels["death_label"] = afib_ecg_labels["death_date"].notna()
afib_ecg_labels["death_time"] = (
    afib_ecg_labels["death_date"].fillna(afib_ecg_labels["censor_time"]) - afib_ecg_labels["ecg_time"]
).dt.days

print(f"Missing censor rows: {missing_censor_count:,}")
print(f"Dropped ECGs without follow-up (ecg_time >= censor_time): {no_followup_count:,}")
print(f"Labelable AFib ECGs: {len(afib_ecg_labels):,}")
for outcome_name, slug in OUTCOME_SLUGS.items():
    print(f"{outcome_name} positives: {int(afib_ecg_labels[f'{slug}_label'].sum()):,}")
print(f"Death positives: {int(afib_ecg_labels['death_label'].sum()):,}")

display(
    afib_ecg_labels[
        [
            "subject_id",
            "study_id",
            "ecg_time",
            "censor_time",
            "intracranial_hemorrhage_label",
            "intracranial_hemorrhage_date",
            "intracranial_hemorrhage_time",
            "gastrointestinal_bleeding_label",
            "gastrointestinal_bleeding_date",
            "gastrointestinal_bleeding_time",
            "other_major_bleeding_label",
            "other_major_bleeding_date",
            "other_major_bleeding_time",
            "death_label",
            "death_date",
            "death_time",
        ]
    ].head()
)


Missing censor rows: 0
Dropped ECGs without follow-up (ecg_time >= censor_time): 2,323
Labelable AFib ECGs: 121,193
Intracranial hemorrhage positives: 2,947
Gastrointestinal bleeding positives: 17,602
Other major bleeding (non-GI / non-intracranial) positives: 21,333
Death positives: 55,403


,subject_id,study_id,ecg_time,censor_time,intracranial_hemorrhage_label,intracranial_hemorrhage_date,intracranial_hemorrhage_time,gastrointestinal_bleeding_label,gastrointestinal_bleeding_date,gastrointestinal_bleeding_time,other_major_bleeding_label,other_major_bleeding_date,other_major_bleeding_time,death_label,death_date,death_time
0,10001877,41316269,2149-05-25 02:44:00,2151-01-30 22:58:00,False,NaT,615,False,NaT,615,False,NaT,615,False,NaT,615
1,10001877,46566322,2150-11-21 18:48:00,2151-01-30 22:58:00,False,NaT,70,False,NaT,70,False,NaT,70,False,NaT,70
2,10001877,42315368,2150-11-21 23:10:00,2151-01-30 22:58:00,False,NaT,69,False,NaT,69,False,NaT,69,False,NaT,69
3,10001884,49736004,2130-04-15 16:55:00,2131-01-20 23:59:59,False,NaT,280,True,2130-08-21 12:14:00,127,True,2130-08-21 12:14:00,127,True,2131-01-20 23:59:59,280
4,10001884,40473394,2130-04-15 20:16:00,2131-01-20 23:59:59,False,NaT,280,True,2130-08-21 12:14:00,127,True,2130-08-21 12:14:00,127,True,2131-01-20 23:59:59,280


In [22]:
afib_ecg_labels['subject_id'].nunique()
afib_ecg_labels.to_parquet("/data/padmalab_external/special_project/Weijie_Code/AFib_labels/afib_ecg_labels.parquet")

In [23]:
HORIZON_DAYS = {
    "30days": 30,
    "1yr": 365,
}


def build_horizon_label(event_label: pd.Series, event_time: pd.Series, horizon_days: int) -> pd.Series:
    out = pd.Series(-1, index=event_time.index, dtype="int8")
    out.loc[event_time > horizon_days] = 0
    out.loc[event_label & (event_time <= horizon_days)] = 1
    return out


binary_label_columns = {}
for outcome_name, slug in {**OUTCOME_SLUGS, "Death": "death"}.items():
    label_col = f"{slug}_label"
    time_col = f"{slug}_time"
    for horizon_name, horizon_days in HORIZON_DAYS.items():
        binary_col = f"{slug}_{horizon_name}"
        binary_label_columns[binary_col] = build_horizon_label(
            afib_ecg_labels[label_col],
            afib_ecg_labels[time_col],
            horizon_days,
        )

afib_ecg_binary_labels = afib_ecg_labels[["subject_id", "study_id", "ecg_time", "path"]].copy()
for col_name, values in binary_label_columns.items():
    afib_ecg_binary_labels[col_name] = values

binary_output_path = "/data/padmalab_external/special_project/Weijie_Code/AFib_labels/afib_ecg_binary_labels.parquet"
afib_ecg_binary_labels.to_parquet(binary_output_path, index=False)

print(f"Saved binary AFib labels to: {binary_output_path}")
for col in [
    "intracranial_hemorrhage_30days",
    "intracranial_hemorrhage_1yr",
    "gastrointestinal_bleeding_30days",
    "gastrointestinal_bleeding_1yr",
    "other_major_bleeding_30days",
    "other_major_bleeding_1yr",
    "death_30days",
    "death_1yr",
]:
    print(col)
    print(afib_ecg_binary_labels[col].value_counts(dropna=False).sort_index())

display(afib_ecg_binary_labels.head())


Saved binary AFib labels to: /data/padmalab_external/special_project/Weijie_Code/AFib_labels/afib_ecg_binary_labels.parquet
intracranial_hemorrhage_30days
intracranial_hemorrhage_30days
-1     16253
 0    104519
 1       421
Name: count, dtype: int64
intracranial_hemorrhage_1yr
intracranial_hemorrhage_1yr
-1    42553
 0    77494
 1     1146
Name: count, dtype: int64
gastrointestinal_bleeding_30days
gastrointestinal_bleeding_30days
-1     16106
 0    102609
 1      2478
Name: count, dtype: int64
gastrointestinal_bleeding_1yr
gastrointestinal_bleeding_1yr
-1    40399
 0    73249
 1     7545
Name: count, dtype: int64
other_major_bleeding_30days
other_major_bleeding_30days
-1     16043
 0    102741
 1      2409
Name: count, dtype: int64
other_major_bleeding_1yr
other_major_bleeding_1yr
-1    40313
 0    73439
 1     7441
Name: count, dtype: int64
death_30days
death_30days
-1     10214
 0    104785
 1      6194
Name: count, dtype: int64
death_1yr
death_1yr
-1    20331
 0    77953
 1    2290

,subject_id,study_id,ecg_time,path,intracranial_hemorrhage_30days,intracranial_hemorrhage_1yr,gastrointestinal_bleeding_30days,gastrointestinal_bleeding_1yr,other_major_bleeding_30days,other_major_bleeding_1yr,death_30days,death_1yr
0,10001877,41316269,2149-05-25 02:44:00,files/p1000/p10001877/s41316269/41316269,0,0,0,0,0,0,0,0
1,10001877,46566322,2150-11-21 18:48:00,files/p1000/p10001877/s46566322/46566322,0,-1,0,-1,0,-1,0,-1
2,10001877,42315368,2150-11-21 23:10:00,files/p1000/p10001877/s42315368/42315368,0,-1,0,-1,0,-1,0,-1
3,10001884,49736004,2130-04-15 16:55:00,files/p1000/p10001884/s49736004/49736004,0,-1,0,1,0,1,0,1
4,10001884,40473394,2130-04-15 20:16:00,files/p1000/p10001884/s40473394/40473394,0,-1,0,1,0,1,0,1


In [25]:
afib_ecg_binary_labels.to_parquet('/data/padmalab_external/special_project/Weijie_Code/AFib_labels/afib_ecg_binary_labels.parquet')